
# Context Engineering – Hands-On Demo

## Objective
In this lab, we will learn how to:
- Structure context instead of dumping prompts
- Separate static, dynamic, and memory context
- Build a real-world, AI system

### Use Case
Customer Support AI for an EdTech platform

> Key idea: **Context is infrastructure, not text.**


## Problem Statement

We want to build an AI system that:
- Acts as a customer support agent
- Follows company refund policies strictly
- Responds politely and clearly
- Uses user-specific information to decide responses

This is **not a chatbot**.  
This is a **context-aware AI system**.

## The Wrong Way: Context Dumping

Many people put everything into one prompt.
This approach does not scale and is hard to control.

In [ ]:
bad_prompt = """
You are a support agent.
Follow all rules.
Here is the refund policy...
User wants a refund.
"""
print(bad_prompt)


You are a support agent.
Follow all rules.
Here is the refund policy...
User wants a refund.



## Context Engineering Approach

We break context into structured parts:

1. Static Context → Who the AI is and how it behaves
2. External Context → Policies or documents
3. Dynamic Context → User input and task data
4. Memory Context → What the system remembers

We assemble these intentionally.

## Step 1: Static Context

Static context defines:
- The role of the AI
- Rules and constraints
- Tone and behavior

This rarely changes.

In [ ]:
SYSTEM_CONTEXT = """
You are a customer support assistant for an EdTech platform.

Rules:
- Be polite and professional
- Do not promise refunds
- Follow company policies strictly
- Escalate to human support if unsure
"""

## Step 2: External Context (Policies)

The AI should not guess policies.
We explicitly provide them as context.

In [ ]:
REFUND_POLICY = """
Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable
"""

## Step 3: Dynamic Context

Dynamic context changes per request.
This includes user input and user-specific data.

In [ ]:
user_query = "I purchased the course 10 days ago and want a refund"

user_profile = {
    "role": "student",
    "course_progress": "15%",
    "purchase_days_ago": 10
}

## Step 4: Context Assembly

We now assemble the context carefully.
Order and clarity matter.
This is **context engineering**.

In [ ]:
final_prompt = f"""
{SYSTEM_CONTEXT}

Company Policy:
{REFUND_POLICY}

User Profile:
- Role: {user_profile['role']}
- Course Progress: {user_profile['course_progress']}
- Purchased: {user_profile['purchase_days_ago']} days ago

User Question:
{user_query}
"""

In [ ]:
print(final_prompt)



You are a customer support assistant for an EdTech platform.

Rules:
- Be polite and professional
- Do not promise refunds
- Follow company policies strictly
- Escalate to human support if unsure


Company Policy:

Refund Policy:
- Refunds are allowed only within 7 days of purchase
- Course progress must be below 20%
- Subscriptions are non-refundable


User Profile:
- Role: student
- Course Progress: 15%
- Purchased: 10 days ago

User Question:
I purchased the course 10 days ago and want a refund



In [ ]:
from google.colab import userdata

# Access the API key stored in Colab Secrets
MY_API_KEY = userdata.get('api_key')

# Now, use this variable when initializing your OpenAI client
# client = OpenAI(api_key=MY_API_KEY, base_url="https://apidev.navigatelabsai.com")

print("API key loaded successfully from Colab Secrets.")
print("Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.")

API key loaded successfully from Colab Secrets.
Please update the client initialization in the cell above (hok-nnOpgi2r) to use `api_key=MY_API_KEY`.


## Step 5: Call the Model

We now send the structured context to the model.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=MY_API_KEY, base_url="https://nexusapi.navigatelabs.ai")

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt}
    ]
)

print(response.choices[0].message.content)

Thank you for reaching out!

I understand you're looking for a refund for your course. I've checked our records and our refund policy. According to our policy, refunds are permitted within 7 days of purchase, provided that course progress is below 20%.

While your course progress is currently 15%, your purchase was made 10 days ago, which falls outside of our 7-day refund window.

Therefore, we are unable to process a refund in this instance.

If you have any other questions or need assistance with your course content, please don't hesitate to ask!


## Why Did the AI Respond Correctly?

The model:
- Refused the refund
- Followed policy
- Maintained a professional tone

This happened **because of context**, not because the model is smart.

## Step 6: Memory Context

Real AI systems remember important user information.
We store **summarized memory**, not full chat history.

In [ ]:
SESSION_MEMORY = """
User previously asked about course difficulty.
User is price-sensitive.
"""

## Context Assembly with Memory

We now include session memory into the context.

In [ ]:
final_prompt_with_memory = f"""
{SYSTEM_CONTEXT}

Session Memory:
{SESSION_MEMORY}

Company Policy:
{REFUND_POLICY}

User Question:
{user_query}
"""

In [ ]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT},
        {"role": "user", "content": final_prompt_with_memory}
    ]
)

print(response.choices[0].message.content)

Hello! Thank you for reaching out.

I understand you're looking for a refund for the course you purchased 10 days ago. Our company policy states that refunds are only available for purchases made within the last 7 days.

Given that your purchase was made 10 days ago, it falls outside of this 7-day window, and unfortunately, we are unable to process a refund in this instance.

If you have any other questions about the course content or need assistance with your learning experience, please don't hesitate to ask!


## Your Task

1. Choose a real-world domain
2. Define:
   - Static context
   - Dynamic context
   - Memory context
3. Assemble context in code
4. Show one real workflow
5. Explain how this scales to your capstone

No generic chatbots allowed.

In [76]:
SYSTEM_CONTEXT = """
You are a customer support assistant for a banking platform.

Rules:
- Be polite and professional
- Do not promise refunds or other banking outcomes
- Follow banking policies strictly
- Do not assume missing customer or transaction information
- Ask clarification questions when required information is missing
- Never hallucinate transaction status, fees, balances, or eligibility
- Do not request sensitive information such as passwords, PINs, CVVs, or OTPs
- Escalate to human support if unsure
"""


In [77]:
REFUND_POLICY = """
Banking Refund Policy:
- Refund requests are allowed only within 7 days of the transaction
- The transaction must be eligible for a refund under the bank's policy
- Cash withdrawals are non-refundable
- Unauthorized transactions must be reported immediately
- Refund eligibility cannot be determined without sufficient transaction details
"""


In [78]:
user_query = "I made this transaction 10 days ago and want a refund"

user_profile = {
    "account_type": "savings",
    "transaction_status": "completed",
    "transaction_days_ago": 10
}


In [79]:
final_prompt = f"""
{SYSTEM_CONTEXT}

Banking Policy:
{REFUND_POLICY}

User Profile:
- Account Type: {user_profile['account_type']}
- Transaction Status: {user_profile['transaction_status']}
- Transaction Date: {user_profile['transaction_days_ago']} days ago

User Question:
{user_query}
"""


In [80]:
print(final_prompt)




You are a customer support assistant for a banking platform.

Rules:
- Be polite and professional
- Do not promise refunds or other banking outcomes
- Follow banking policies strictly
- Do not assume missing customer or transaction information
- Ask clarification questions when required information is missing
- Never hallucinate transaction status, fees, balances, or eligibility
- Do not request sensitive information such as passwords, PINs, CVVs, or OTPs
- Escalate to human support if unsure


Banking Policy:

Banking Refund Policy:
- Refund requests are allowed only within 7 days of the transaction
- The transaction must be eligible for a refund under the bank's policy
- Cash withdrawals are non-refundable
- Unauthorized transactions must be reported immediately
- Refund eligibility cannot be determined without sufficient transaction details


User Profile:
- Account Type: savings
- Transaction Status: completed
- Transaction Date: 10 days ago

User Question:
I made this transactio

In [82]:
from google.colab import userdata

# Access the API key stored in Colab Secrets
MY_API_KEY = userdata.get("api_key")

# Verify that the API key was loaded
if MY_API_KEY:
    print("API key loaded successfully from Colab Secrets.")
else:
    print("API key was not found. Please check your Colab Secrets.")


API key loaded successfully from Colab Secrets.


In [84]:
from openai import OpenAI

client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt
        }
    ]
)

print(response.choices[0].message.content)


Thank you for reaching out.

Regarding your refund request, our banking policy states that refund requests are allowed within 7 days of the transaction date. As your transaction occurred 10 days ago, it falls outside of this 7-day window.

To further assess any available options or to provide more specific assistance, please provide additional details about the transaction, such as the merchant name, amount, and transaction type. However, please be aware that the 7-day timeframe for refund requests is a key factor.


In [85]:
SESSION_MEMORY = """
User previously asked about transaction refund policies.
User prefers clear and concise explanations.
User has previously asked about banking transaction charges.
"""


In [90]:
final_prompt_with_memory = f"""
{SYSTEM_CONTEXT}

Session Memory:
{SESSION_MEMORY}

Banking Policy:
{REFUND_POLICY}

User Question:
{user_query}
"""


In [92]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT
        },
        {
            "role": "user",
            "content": final_prompt_with_memory
        }
    ]
)

print(response.choices[0].message.content)


I understand you're looking to request a refund for a transaction.

According to our banking policy, refund requests are allowed within 7 days of the transaction date. As your transaction was made 10 days ago, it falls outside of this 7-day window.

If you have any further questions or if this transaction was unauthorized, please let me know.
